# EECS 182 - Fall 2026
## Homework 02: Watching Momentum Work

This notebook supports parts (h) and (i) of **Accelerating Gradient Descent with Momentum**
(Problem 2 in Homework 02, originally Problem 4 in the Homework 1 bank), and
**Coding: Watching Momentum Work** (Problem 5).

Contributors: Matteo Guarrera, Mert Cemri, Sizhe Chen, Suhong Moon, Gabriel Goh,
Anant Sahai, Peter Wang, Yuxi Liu.

[Open in Colab](https://colab.research.google.com/github/Berkeley-CS182/cs182fa26_public/blob/main/hw02/code/q_sgd_momentum_analysis.ipynb)

Run all cells in order using a Python 3 notebook or a standard Colab CPU runtime.
Only NumPy and Matplotlib are needed. All data and optimizer code are included.
Answer the two observation questions once in your written homework; Problem 5 may refer to your answers to Problem 2(h) and (i). No notebook submission is required.


### The exact example from part (g)

We minimize the **sum** of squared errors, $L(w)=\|Xw-y\|_2^2$.
The singular values are exactly $2$ and $1$, the momentum parameter is
$\beta=3/4$, and we compare $\eta_{\mathrm{GD}}=1/5$ with
$\eta_{\mathrm{momentum}}=1/3$.

Our four data points have $X^\top X=\operatorname{diag}(4,1)$, so the two
coordinate axes are exactly the singular directions. This lets us connect each
plotted coordinate directly to the analysis in the written problem.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

X = np.array([[1.0, 0.5], [1.0, -0.5], [-1.0, 0.5], [-1.0, -0.5]])
w_star = np.array([1.0, 1.0])
y = X @ w_star
gram = X.T @ X
singular_values = np.linalg.svd(X, compute_uv=False)
loss_star = float(np.sum((X @ w_star - y)**2))

BETA = 3 / 4
ETA_GD = 1 / 5
ETA_MOMENTUM = 1 / 3
STEPS = 20

assert np.allclose(gram, np.diag([4.0, 1.0]))
assert np.allclose(singular_values, [2.0, 1.0])
print('X.T @ X =\n', gram)
print('Singular values:', singular_values)
print('Exact minimizer:', w_star, '| Minimum loss:', loss_star)
print(f'beta = {BETA:g}; eta_GD = {ETA_GD:g}; eta_momentum = {ETA_MOMENTUM:g}')


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
points = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=130,
                    edgecolors='black', vmin=-1.5, vmax=1.5)
for point, label in zip(X, y):
    ax.annotate(f'y={label:g}', point, xytext=(7, 8), textcoords='offset points')
fig.colorbar(points, ax=ax, label='Regression target y')
ax.set(xlabel='Feature 0', ylabel='Feature 1', title='Four deterministic regression samples',
       xlim=(-1.6, 1.6), ylim=(-0.9, 0.9))
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()


### The optimizer and its initial state

Both runs start from $w_0=(0,0)$ and $a_0=(0,0)$.
At each iteration, we use the same convention as the homework:

$$g_t=2X^\top(Xw_t-y),\qquad
a_{t+1}=(1-\beta)a_t+\beta g_t,\qquad
w_{t+1}=w_t-\eta a_{t+1}.$$

Setting $\beta=1$ gives ordinary gradient descent. In particular, the first
momentum update uses $a_1=\beta g_0$. The trace below includes the initial state
at iteration 0 and then every state after an update.


In [ ]:
def run_descent(eta, beta=1.0, steps=STEPS):
    if eta <= 0 or not 0 < beta <= 1 or steps < 1:
        raise ValueError('Use eta > 0, 0 < beta <= 1, and at least one step.')
    w = np.zeros(X.shape[1])
    a = np.zeros_like(w)
    parameters = [w.copy()]
    averaged_gradients = [a.copy()]
    for _ in range(steps):
        gradient = 2 * X.T @ (X @ w - y)
        a = (1 - beta) * a + beta * gradient
        w = w - eta * a
        parameters.append(w.copy())
        averaged_gradients.append(a.copy())
    parameters = np.asarray(parameters)
    errors = parameters - w_star
    residuals = parameters @ X.T - y
    losses = np.sum(residuals**2, axis=1)
    gradients = 2 * residuals @ X
    return dict(parameters=parameters, errors=errors, gradients=gradients,
                averaged_gradients=np.asarray(averaged_gradients), losses=losses,
                relative_error=np.linalg.norm(errors, axis=1) / np.linalg.norm(errors[0]),
                eta=eta, beta=beta)

gd = run_descent(ETA_GD)
momentum = run_descent(ETA_MOMENTUM, BETA)
iterations = np.arange(STEPS + 1)
for name, trace in [('GD', gd), ('Momentum', momentum)]:
    print(f'{name:8s}: w_{STEPS} = {trace["parameters"][-1]}, '
          f'loss = {trace["losses"][-1]:.3e}')


### Watch each singular direction

The plots show **signed** parameter errors and gradients, so crossing zero is
visible. A parameter error of zero means that coordinate has reached its
optimal value. Coordinate 0 has $\sigma=2$; coordinate 1 has $\sigma=1$.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for dimension, sigma in enumerate(singular_values):
    for name, trace, color in [('GD', gd, '#2166ac'), ('Momentum', momentum, '#d6604d')]:
        axes[0, dimension].plot(iterations, trace['errors'][:, dimension],
                                'o-', markersize=3, color=color, label=name)
        axes[1, dimension].plot(iterations, trace['gradients'][:, dimension],
                                'o-', markersize=3, color=color, label=name)
    axes[0, dimension].set_title(f'Coordinate {dimension}: singular value {sigma:g}')
    axes[0, dimension].set_ylabel('Signed parameter error')
    axes[1, dimension].set_ylabel('Signed gradient')
    axes[1, dimension].set_xlabel('Iteration t')
for ax in axes.flat:
    ax.axhline(0, color='black', linewidth=0.7)
    ax.grid(alpha=0.2)
    ax.legend()
fig.tight_layout()
plt.show()


### Written question (h)

How does the singular value $\sigma_i$ influence the gradients and the
parameter updates along direction $i$? Use the plots to support your answer.


### Parameter paths on the loss landscape

The contour plot shows the early iterates. Use the signed-coordinate plots
above to examine individual crossings of the optimum.


In [ ]:
w0_grid, w1_grid = np.meshgrid(np.linspace(-0.15, 2.15, 150),
                              np.linspace(-0.15, 1.4, 150))
grid_loss = 4 * (w0_grid - 1)**2 + (w1_grid - 1)**2
fig, ax = plt.subplots(figsize=(8, 5))
contours = ax.contour(w0_grid, w1_grid, grid_loss,
                     levels=[0.01, 0.04, 0.16, 0.64, 1.5, 3, 5],
                     colors='0.7', linewidths=0.8)
ax.clabel(contours, inline=True, fontsize=8)
for name, trace, color in [('GD', gd, '#2166ac'), ('Momentum', momentum, '#d6604d')]:
    path = trace['parameters'][:9]
    ax.plot(path[:, 0], path[:, 1], 'o-', markersize=4, color=color, label=name)
ax.plot(*w_star, '*', color='black', markersize=13, label='Exact optimum')
ax.set(xlabel='w[0]', ylabel='w[1]', title='First 8 updates on the loss contours')
ax.legend()
fig.tight_layout()
plt.show()


### Measured errors and geometric factors

The first panel compares the actual relative parameter errors. The second
compares the geometric factors $r^t$ from part (g). A spectral radius describes
the asymptotic geometric factor; momentum can also have an initial transient.

The loss plot uses the known optimum $L(w^*)=0$. We apply a small positive floor
only for display on a logarithmic axis, so zeros never produce an invalid log.


In [ ]:
def system_matrix(sigma, eta, beta):
    return np.array([[1 - beta, 2 * beta * sigma**2],
                     [-eta * (1 - beta), 1 - 2 * eta * beta * sigma**2]])

rate_gd = float(np.max(np.abs(1 - 2 * ETA_GD * singular_values**2)))
rate_momentum = max(float(np.max(np.abs(np.linalg.eigvals(
    system_matrix(sigma, ETA_MOMENTUM, BETA))))) for sigma in singular_values)
print(f'GD geometric factor: {rate_gd:g}; Momentum geometric factor: {rate_momentum:g}')

floor = 1e-16  # Display floor only; original traces are left unchanged.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name, trace, rate, color in [('GD', gd, rate_gd, '#2166ac'),
                                ('Momentum', momentum, rate_momentum, '#d6604d')]:
    axes[0].semilogy(iterations, np.maximum(trace['relative_error'], floor),
                     'o-', markersize=3, color=color, label=name)
    axes[1].semilogy(iterations, rate**iterations, color=color, label=f'{name}: r={rate:g}')
    axes[2].semilogy(iterations, np.maximum(trace['losses'] - loss_star, floor),
                     'o-', markersize=3, color=color, label=name)
axes[0].set(title='Measured relative parameter error', ylabel='Relative error')
axes[1].set(title='Geometric factors from part (g)', ylabel='r to the power t')
axes[2].set(title='Measured loss above the optimum', ylabel='L(w_t) - L(w*)')
for ax in axes:
    ax.set_xlabel('Iteration t')
    ax.grid(alpha=0.2)
    ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
def first_at_most(values, threshold):
    hits = np.flatnonzero(np.asarray(values) <= threshold)
    return int(hits[0]) if hits.size else None

target = 1 / 4
print(f'Target: {target:g}')
print('Method     Geometric-factor count    Measured relative-error count')
for name, trace, rate in [('GD', gd, rate_gd), ('Momentum', momentum, rate_momentum)]:
    geometric_count = first_at_most(rate**iterations, target + 1e-14)
    actual_count = first_at_most(trace['relative_error'], target)
    print(f'{name:10s} {geometric_count!s:25s} {actual_count}')
print('These are different comparisons: r^t and the measured parameter error.')


### Written question (i)

Comparing gradient descent with and without momentum, which converges faster
on this task, and why? Use the plots to support your answer.

The geometric factors and measured trajectories help explain the long-run
behavior as well as the first few steps. Record your observations in the
written homework.


### Optional exploration

You can edit `ETA_MOMENTUM` and rerun the notebook to see how the learning rate
changes the trajectories. Part (g)'s optimal interval is $[1/6,3/8]$.
At its endpoints one mode has repeated roots; an interior value such as $1/3$
gives complex-conjugate roots in both modes. This is optional exploration and
adds no submission requirement.

The [companion visualization](https://colab.research.google.com/github/Berkeley-CS182/cs182fa26_public/blob/main/hw02/code/q_sgd_momentum_analysis_visualization.ipynb)
shows the eigenvalues moving as the learning rate changes.
